# Notebook 02 — Preparação para Modelagem

Parte da base tratada produzida no notebook 01 e a prepara para os modelos:
separação por finalidade, divisão treino/teste, transformações e encoding. O
princípio que rege esta fase é a **prevenção de vazamento (leakage)**: a divisão
treino/teste é feita logo no início, e toda transformação que aprende algo dos
dados (estatísticas, parâmetros de encoding) será ajustada **apenas no treino**.

In [1]:
import pandas as pd

df = pd.read_parquet("data/processed/imoveis_tratados.parquet")

print("Base tratada carregada:", df.shape)
df.info()

Base tratada carregada: (12759, 16)
<class 'pandas.DataFrame'>
Index: 12759 entries, 0 to 13639
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Price             12759 non-null  int64  
 1   Condo             10872 non-null  float64
 2   Size              12759 non-null  int64  
 3   Rooms             12759 non-null  int64  
 4   Toilets           12759 non-null  int64  
 5   Suites            12759 non-null  int64  
 6   Parking           12759 non-null  int64  
 7   Elevator          12759 non-null  int64  
 8   Furnished         12759 non-null  int64  
 9   Swimming Pool     12759 non-null  int64  
 10  New               12759 non-null  int64  
 11  District          12759 non-null  str    
 12  Negotiation Type  12759 non-null  str    
 13  Property Type     12759 non-null  str    
 14  Latitude          12759 non-null  float64
 15  Longitude         12759 non-null  float64
dtypes: float64(3), int64

In [2]:
# Separa a base de dados tratada entre as finalidades de venda e locação
df_venda = df[df["Negotiation Type"] == "sale"].copy()
df_aluguel = df[df["Negotiation Type"] == "rent"].copy()

print("Venda:  ", df_venda.shape)
print("Aluguel:", df_aluguel.shape)
print("Soma:   ", len(df_venda) + len(df_aluguel))

Venda:   (6014, 16)
Aluguel: (6745, 16)
Soma:    12759


## 1. Divisão treino/teste

Antes de qualquer transformação, separamos cada base (venda e aluguel) em treino e
teste. Esta ordem é deliberada e inegociável: tudo que o modelo aprende — incluindo
estatísticas de imputação, parâmetros de encoding e o spatial lag — será calculado
**somente no treino** e aplicado ao teste. O conjunto de teste representa imóveis
"futuros", nunca vistos pelo processo de modelagem; deixá-lo influenciar qualquer
etapa do aprendizado seria vazamento (leakage), inflando artificialmente o
desempenho e mascarando como o modelo se comportaria na realidade.

Usamos divisão de 80% treino / 20% teste, com seed fixa para reprodutibilidade.

In [3]:
from sklearn.model_selection import train_test_split

# Separa as features (X) do alvo (y) em cada base.

def separar_treino_teste(dados, nome):
    X = dados.drop(columns="Price")   # tudo, menos o alvo
    y = dados["Price"]                # o alvo
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.20,      # 20% para teste
        random_state=42      # semente fixa: a divisão é sempre a mesma
    )
    print(f"{nome:8} → treino: {X_train.shape[0]:>5} | teste: {X_test.shape[0]:>5}")
    return X_train, X_test, y_train, y_test

X_train_venda, X_test_venda, y_train_venda, y_test_venda = separar_treino_teste(df_venda, "Venda")
X_train_alug,  X_test_alug,  y_train_alug,  y_test_alug  = separar_treino_teste(df_aluguel, "Aluguel")

Venda    → treino:  4811 | teste:  1203
Aluguel  → treino:  5396 | teste:  1349


Cada base foi dividida em 80% treino / 20% teste com semente fixa (`random_state=42`),
garantindo que a divisão seja idêntica a cada execução. A partir daqui, **o conjunto
de teste fica intocado** até a avaliação final dos modelos: nenhuma estatística,
transformação ou decisão de modelagem o consultará. Toda a preparação a seguir é
ajustada sobre o treino e apenas aplicada ao teste.

## 2. Definição do conjunto de variáveis (candidato de partida)

Antes de transformar qualquer coluna, fixamos quais variáveis entram no modelo, por
quê, e quais ficam de fora. Esta é a primeira de três decisões de variáveis no
projeto: aqui definimos o **conjunto candidato de partida** (o que a base já oferece);
as variáveis espaciais e socioeconômicas serão **criadas** na Fase 3; e o conjunto
**final** será decantado empiricamente na modelagem (via VIF e importância de
features). Esta decisão é estrutural e baseada em domínio — não consulta o conjunto
de teste —, portanto não viola a regra anti-leakage.

**Alvo:** `log(Price)`. A relação hedônica é multiplicativa e o preço tem forte
assimetria à direita (média ≫ mediana, vista na auditoria); o log lineariza a relação
e normaliza os resíduos, como a NBR 14653 pressupõe.

**Atributos físicos:**
- `log(Size)` — logado pelo mesmo motivo do preço: em log-log, o coeficiente vira uma
  **elasticidade** (1% a mais de área → β% a mais de preço), forma funcional correta
  do preço hedônico, com retorno marginal decrescente por m². A área também é tão
  assimétrica quanto o preço.
- `Rooms`, `Toilets`, `Suites`, `Parking` — contagens, entram diretas. Atenção
  registrada: são correlacionadas entre si (risco de multicolinearidade), a ser
  checado por VIF na modelagem.

**Comodidades / binárias:** `Elevator`, `Furnished`, `Swimming Pool`, `New`. Entram
como 0/1. Ressalva: `New` (97–99% zero) e `Furnished` têm variância baixíssima —
candidatas a sair se não contribuírem.

**Localização (baseline):** `District` → dummies (one-hot). Captura o nível médio de
preço por bairro. É a localização "grossa"; a localização fina (spatial lag,
distância, socioeconômico) será adicionada na Fase 3, permitindo medir o ganho do
tratamento espacial sobre este baseline.

**Fora do modelo:**
- `Condo` — descartado por três razões: (1) ~15% de valores ausentes; (2) risco de
  multicolinearidade com `Swimming Pool`, academia e quadra, que preferimos manter por
  estarem completas; (3) o missing é provavelmente **não-aleatório** — comodidades
  costumam ser divulgadas, mas condomínio alto em reais tende a ser omitido no
  anúncio, então o próprio `NaN` carrega viés, e imputá-lo distorceria. Aposta
  assumida: piscina/academia/quadra capturam o sinal de qualidade que o condomínio
  traria.
- `Property Type` — constante (100% apartamentos), sem poder explicativo.
- `Negotiation Type` — é o critério de separação das duas bases, não uma feature.
- `Latitude` / `Longitude` crus — não entram diretamente; são **insumo** para as
  features espaciais da Fase 3.